# Movies Data — Exploratory Analysis

**Purpose:** understand `movies_data_assignment.csv` before designing the MongoDB schema,
the ingestion pipeline, and the query API.

This notebook is exploration-only. The CSV is gitignored and is **not** part of the
deliverable. Nothing here ships to production — it just informs design decisions.

Each section ends with the design question it answers.

## 0. Setup

In [1]:
import os
import ast
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# CSV sits next to this notebook in test_scripts/
CSV_PATH = "movies_data_assignment.csv"
assert os.path.exists(CSV_PATH), f"CSV not found at {CSV_PATH!r} — adjust the path"


## 1. Shape & size — how big is "1GB" really?

The sample is small, but we can estimate bytes-per-row and extrapolate to the 1GB
worst case the assignment specifies. That number is what justifies async processing
and batched inserts.

In [2]:
df = pd.read_csv(CSV_PATH)
print("Rows   :", len(df))
print("Columns:", df.shape[1])
df.head()


Rows   : 45428
Columns: 15


,budget,homepage,original_language,original_title,overview,release_date,revenue,runtime,status,title,vote_average,vote_count,production_company_id,genre_id,languages
0,30000000.0,http://toystory.disney.com/toy-story,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",1995-10-30,373554033.0,81,Released,Toy Story,7.7,5415.0,3,16,['English']
1,65000000.0,NaN,en,Jumanji,When siblings Judy and Peter discover an encha...,1995-12-15,262797249.0,104,Released,Jumanji,6.9,2413.0,559,12,"['English', 'Français']"
2,0.0,NaN,en,Grumpier Old Men,A family wedding reignites the ancient feud be...,1995-12-22,0.0,101,Released,Grumpier Old Men,6.5,92.0,6194,10749,['English']
3,16000000.0,NaN,en,Waiting to Exhale,"Cheated on, mistreated and stepped on, the wom...",1995-12-22,81452156.0,127,Released,Waiting to Exhale,6.1,34.0,306,35,['English']
4,0.0,NaN,en,Father of the Bride Part II,Just when George Banks has recovered from his ...,1995-02-10,76578911.0,106,Released,Father of the Bride Part II,5.7,173.0,5842,35,['English']


In [3]:
size_bytes = os.path.getsize(CSV_PATH)
bytes_per_row = size_bytes / max(len(df), 1)
rows_in_1gb = 1_073_741_824 / bytes_per_row

print(f"File size on disk : {size_bytes/1_048_576:.2f} MB")
print(f"Approx bytes/row  : {bytes_per_row:.0f}")
print(f"Est. rows in 1 GB : {rows_in_1gb:,.0f}")
print()
print("In-memory footprint (deep):")
df.info(memory_usage="deep")


File size on disk : 18.74 MB
Approx bytes/row  : 433
Est. rows in 1 GB : 2,482,040

In-memory footprint (deep):
<class 'pandas.DataFrame'>
RangeIndex: 45428 entries, 0 to 45427
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   budget                 45428 non-null  float64
 1   homepage               7772 non-null   str    
 2   original_language      45417 non-null  str    
 3   original_title         45428 non-null  str    
 4   overview               44474 non-null  str    
 5   release_date           45344 non-null  str    
 6   revenue                45428 non-null  float64
 7   runtime                45428 non-null  int64  
 8   status                 45347 non-null  str    
 9   title                  45428 non-null  str    
 10  vote_average           45428 non-null  float64
 11  vote_count             45428 non-null  float64
 12  production_company_id  45428 non-null  int64  
 13  genre

## 2. Columns & dtypes

In [4]:
df.dtypes.to_frame("dtype")


,dtype
budget,float64
homepage,str
original_language,str
original_title,str
overview,str
release_date,str
revenue,float64
runtime,int64
status,str
title,str


## 3. Missing values

Decides which fields are required vs optional in validation, and how the pipeline
should treat empty cells.

In [5]:
missing = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
}).sort_values("null_count", ascending=False)
missing


,null_count,null_pct
homepage,37656,82.89
overview,954,2.10
release_date,84,0.18
status,81,0.18
original_language,11,0.02
original_title,0,0.00
budget,0,0.00
revenue,0,0.00
runtime,0,0.00
title,0,0.00


## 4. `release_date` — the field both a filter (year) and a sort depend on

Check: how many values are empty, how many are non-empty but unparseable, the date
range, and how many distinct years exist (index selectivity for the year filter).

In [6]:
raw_missing = df["release_date"].isna().sum()
parsed = pd.to_datetime(df["release_date"], errors="coerce")
non_empty_unparseable = parsed.isna().sum() - raw_missing

print("Empty/missing release_date      :", raw_missing)
print("Non-empty but NOT parseable     :", non_empty_unparseable)
print("Min date                        :", parsed.min())
print("Max date                        :", parsed.max())

df["_year"] = parsed.dt.year
print("Distinct years                  :", df["_year"].nunique())
print()
print("Rows per year (most recent 15):")
df["_year"].value_counts().sort_index().tail(15)


Empty/missing release_date      : 84
Non-empty but NOT parseable     : 0
Min date                        : 1874-12-09 00:00:00
Max date                        : 2020-12-16 00:00:00
Distinct years                  : 135

Rows per year (most recent 15):


_year
2005.0    1124
2006.0    1269
2007.0    1319
2008.0    1470
2009.0    1585
2010.0    1501
2011.0    1666
2012.0    1720
2013.0    1887
2014.0    1973
2015.0    1904
2016.0    1604
2017.0     532
2018.0       5
2020.0       1
Name: count, dtype: int64

## 5. Language fields — which one do we filter on?

There are two candidates:
- `original_language` — a single ISO code (e.g. `en`)
- `languages` — a stringified list (e.g. `['English', 'Français']`)

Look at cardinality of each, and confirm whether `languages` parses cleanly.

In [7]:
print("Distinct original_language values:", df["original_language"].nunique())
print()
df["original_language"].value_counts().head(20)


Distinct original_language values: 89



original_language
en    32247
fr     2436
it     1529
ja     1346
de     1079
es      994
ru      826
hi      508
ko      444
zh      409
sv      383
pt      316
cn      313
fi      295
nl      248
da      224
pl      219
tr      150
cs      130
el      113
Name: count, dtype: int64

In [8]:
def parse_langs(v):
    if pd.isna(v):
        return None
    try:
        return ast.literal_eval(v)
    except (ValueError, SyntaxError):
        return "PARSE_FAILED"

parsed_langs = df["languages"].apply(parse_langs)
n_fail = (parsed_langs == "PARSE_FAILED").sum()
print("languages parse failures:", n_fail)
print("Sample parsed values:")
print(parsed_langs.dropna().head().to_list())

all_langs = set()
for v in parsed_langs:
    if isinstance(v, list):
        all_langs.update(v)
print()
print("Distinct languages across all rows:", len(all_langs))
print(sorted(all_langs)[:30])


languages parse failures: 0
Sample parsed values:
[['English'], ['English', 'Français'], ['English'], ['English'], ['English']]

Distinct languages across all rows: 75
['', '?????', '??????', 'Afrikaans', 'Azərbaycan', 'Bahasa indonesia', 'Bahasa melayu', 'Bamanankan', 'Bokmål', 'Bosanski', 'Català', 'Cymraeg', 'Dansk', 'Deutsch', 'Eesti', 'English', 'Español', 'Esperanto', 'Français', 'Fulfulde', 'Gaeilge', 'Galego', 'Hausa', 'Hrvatski', 'Italiano', 'Kinyarwanda', 'Kiswahili', 'Latin', 'Latviešu', 'Lietuvi\x9akai']


## 6. Rating fields — `vote_average` (sort target) and `vote_count`

`vote_average` is the "rating" the API must sort by. Confirm its range and nulls.

In [9]:
print("vote_average:")
print(df["vote_average"].describe())
print("nulls:", df["vote_average"].isna().sum(),
      "| min:", df["vote_average"].min(), "| max:", df["vote_average"].max())
print()
print("vote_count:")
print(df["vote_count"].describe())


vote_average:
count    45428.000000
mean         5.618299
std          1.924171
min          0.000000
25%          5.000000
50%          6.000000
75%          6.800000
max         10.000000
Name: vote_average, dtype: float64
nulls: 0 | min: 0.0 | max: 10.0

vote_count:
count    45428.000000
mean       109.833275
std        490.975739
min          0.000000
25%          3.000000
50%         10.000000
75%         34.000000
max      14075.000000
Name: vote_count, dtype: float64


## 7. Numeric fields — `budget`, `revenue`, `runtime`

Lots of zeros are expected here. Decide whether `0` means "genuinely zero" or
"unknown" — it affects whether you store 0 or null.

In [10]:
for col in ["budget", "revenue", "runtime"]:
    print(f"--- {col} ---")
    print(df[col].describe())
    print("zeros:", int((df[col] == 0).sum()),
          "| negatives:", int((df[col] < 0).sum()),
          "| nulls:", int(df[col].isna().sum()))
    print()


--- budget ---
count    4.542800e+04
mean     4.224552e+06
std      1.742873e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      3.800000e+08
Name: budget, dtype: float64
zeros: 36549 | negatives: 0 | nulls: 0

--- revenue ---
count    4.542800e+04
mean     1.120885e+07
std      6.434705e+07
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      2.787965e+09
Name: revenue, dtype: float64
zeros: 38031 | negatives: 0 | nulls: 0

--- runtime ---
count    45428.000000
mean        93.590627
std         38.952207
min          0.000000
25%         85.000000
50%         95.000000
75%        107.000000
max       1256.000000
Name: runtime, dtype: float64
zeros: 1815 | negatives: 0 | nulls: 0



## 8. Categorical fields — `status`, `genre_id`, `production_company`

In [11]:
print("status values:")
print(df["status"].value_counts(dropna=False))
print()
print("genre_id distinct          :", df["genre_id"].nunique())
print("production_company distinct:", df["production_company"].nunique())


status values:
status
Released           44983
Rumored              229
Post Production       98
NaN                   81
In Production         20
Planned               15
Canceled               2
Name: count, dtype: int64

genre_id distinct          : 21


KeyError: 'production_company'

## 9. Duplicates — picking a dedup key

There is no unique ID column. To avoid duplicating data on re-upload we need a natural
key. Check how unique `title` alone is vs `(title, release_date)`.

In [12]:
print("Fully duplicated rows        :", int(df.duplicated().sum()))
print("Duplicate title              :", int(df["title"].duplicated().sum()))
print("Duplicate (title, release_date):",
      int(df.duplicated(subset=["title", "release_date"]).sum()))


Fully duplicated rows        : 0
Duplicate title              : 3154
Duplicate (title, release_date): 0


## 10. Summary — design decisions to lock in

Fill these in after running the cells above:

| Decision | Finding | Choice |
|---|---|---|
| Est. rows at 1GB | _from §1_ | → async + batched inserts? |
| Required fields | _from §3_ | which fields must be non-null |
| `release_date` parse failures | _from §4_ | skip row / null the date? |
| Year filter source | _from §4_ | store derived `year:int` |
| Language filter column | _from §5_ | `original_language` (primary) |
| `languages` storage | _from §5_ | parsed array vs raw string |
| `0` in budget/revenue | _from §7_ | keep 0 vs convert to null |
| Dedup key | _from §9_ | `(title, release_date)`? |

These feed directly into the MongoDB schema, the validation layer, and the index plan.